In [107]:
import torch
import torch.nn as nn

In [108]:
def _k(h_a, h_b, lag):
    in_dim = 7
    hidden_dim = in_dim 
    net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1))
    # h_a, h_b: (..., N), lag: (...) -> (...)
    x = torch.cat([(h_a - h_b).abs(), h_a + h_b, lag.unsqueeze(-1)], dim=-1)
    print(x)
    return net(x.squeeze(-1))

In [ ]:
a = torch.tensor([[1,2,3], [4,5,6]], dtype=torch.float)
(M,N) = a.size()
a1 = a.unsqueeze(0).expand(M,M,N)
a2 = a.unsqueeze(1).expand(M,M,N)
lag = torch.tensor([0,1], dtype=torch.float)

print(a1)
print(a2)


tdiff = (lag.unsqueeze(0).expand(M,M) - lag.unsqueeze(1).expand(M,M)).abs()
k = _k(a1, a2, tdiff)

print(k)
print(k.shape)

In [ ]:
import math
h_a = torch.tensor([[1,2,3]], dtype=torch.float)
h_b = torch.tensor([[1,2,3], [4,5,6]], dtype=torch.float)
elapsed_a = torch.tensor([1,2], dtype=torch.float)
elapsed_b = torch.tensor([1,2], dtype=torch.float)

Ma, Na = h_a.shape
Mb, Nb = h_b.shape

print(f'Ma = {Ma}, Mb = {Mb}, Na = {Na}, Nb = {Nb}, h_a.shape = {h_a.shape}, h_b.shape = {h_b.shape}, elapsed.shape = {elapsed.shape}')

assert(Na == Nb)
assert elapsed.numel() == Na, "need one elapsed value per history row"

Mmax = max(Ma, Mb)

# ---- C: all pairs (i, j) of history measurements ----
h_a = h_a.unsqueeze(1).expand(Mmax, Mmax, N)        # row i
h_b = h_b.unsqueeze(0).expand(Mmax, Mmax, N)        # row j
lag = (elapsed.unsqueeze(1) - elapsed.unsqueeze(0)).abs()  # |t_i - t_j|
C = _k(h_a, h_b, lag)                       # (M, M)

n_elem = math.prod(list(C.size()))
print(C.reshape((int(math.sqrt(n_elem)), int(math.sqrt(n_elem)))))

Ma = 1, Mb = 2, Na = 3, Nb = 3, h_a.shape = torch.Size([1, 3]), h_b.shape = torch.Size([2, 3]), elapsed.shape = torch.Size([2])


AssertionError: need one elapsed value per history row

In [ ]:
h_pred = torch.tensor([1,2,3])
h_pred.reshape(1, -1).to(a).expand(M, N)

In [ ]:
a = torch.ones(4, 3, 2)

b = a * torch.rand(3, 2)  # 3rd & 2nd dims identical to a, dim 1 absent
print(b)


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class f_cov(nn.Module):
    """
    Deep kernel, PSD by construction:
      k = s2 * exp(-0.5 * ||phi(h_a) - phi(h_b)||^2) * exp(-|t_a - t_b| / ell_t)

    fcov(H_hist, elapsed)                -> C, shape (M, M)   (includes nugget)
    fcov(H_hist, elapsed, h_pred=h_pred) -> g, shape (M,)
    self.prior_var()                     -> k(h*, h*, 0) = s2
    """

    def __init__(self, in_dim, feat_dim=8, hidden_dim=64, dropout=0.0):
        super().__init__()
        self.phi = nn.Sequential(                      # learned feature map
            nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, feat_dim),
        )
        # unconstrained params -> softplus makes them positive
        self.raw_amp   = nn.Parameter(torch.tensor(0.0))    # signal variance s2
        self.raw_ell_t = nn.Parameter(torch.tensor(0.0))    # time length-scale
        self.raw_noise = nn.Parameter(torch.tensor(-3.0))   # diagonal nugget

    def prior_var(self):
        return F.softplus(self.raw_amp)

    def _k(self, z_a, z_b, lag):
        # z_a: (Ma, D), z_b: (Mb, D), lag: (Ma, Mb) -> (Ma, Mb)
        ell_t = F.softplus(self.raw_ell_t) + 1e-3
        d2 = (z_a[:, None, :] - z_b[None, :, :]).pow(2).sum(-1)  # squared distance
        return self.prior_var() * torch.exp(-0.5 * d2) * torch.exp(-lag / ell_t)

    def forward(self, h_hist, elapsed, h_pred=None):
        if h_hist.dim() == 1:
            h_hist = h_hist.unsqueeze(0)                          # (M, N)
        h_hist = h_hist.float()
        elapsed = elapsed.reshape(-1).to(h_hist)                  # (M,)
        M = h_hist.shape[0]
        assert elapsed.numel() == M, "need one elapsed value per history row"

        z_hist = self.phi(h_hist)                                 # (M, D)

        if h_pred is None:
            lag = (elapsed[:, None] - elapsed[None, :]).abs()     # |t_i - t_j|
            C = self._k(z_hist, z_hist, lag)                      # PSD
            noise = F.softplus(self.raw_noise) + 1e-6
            return C + noise * torch.eye(M, dtype=C.dtype, device=C.device)  # strictly PD

        z_pred = self.phi(h_pred.reshape(1, -1).to(h_hist))       # (1, D)
        return self._k(z_pred, z_hist, elapsed[None, :]).squeeze(0)  # (M,), lag = t* - t_j

In [ ]:
fcov = f_cov()